In [ ]:
from quantbullet.utils.data import generate_fake_bond_trades
from quantbullet.reporting import ExcelExporter, ColumnFormat, ColumnMeta, ColumnSchema
from datetime import date

In [ ]:
data = generate_fake_bond_trades( start_date=date(2020, 1, 1),
                                  end_date=date(2020, 1, 31 ) )

data = data.rename(columns={'ticker': 'Security Name',
                     'date': 'Trade Date',
                     'rating' : 'Original Rating',
                     'expiry' : 'Reinvestment End Date',
                     'yield' : 'Discount Margin',
})
data['feature_B'] = data['feature_B'] * 10e6

In [ ]:
# Only the schema's columns are written, in schema order; display_name defaults to name
schema = ColumnSchema([
    ColumnMeta('Trade Date'),
    ColumnMeta('Security Name', format=ColumnFormat( width=15 )),
    ColumnMeta('Original Rating', format=ColumnFormat( width=10 )),
    ColumnMeta('Discount Margin', format=ColumnFormat( width=10, decimals=0, transformer=lambda x: x * 100 )),
    ColumnMeta('feature_A', format=ColumnFormat( decimals = 1, color_scale=True, higher_is_better=True )),
    ColumnMeta('feature_B', format=ColumnFormat( decimals = 0, comma=True )),
    ColumnMeta('feature_C', format=ColumnFormat( decimals = 1, parens_for_negative=True, color_scale=True, higher_is_better=False )),
    ColumnMeta('feature_D', format=ColumnFormat( decimals = 0, transformer=lambda x: x * 100 )),
    ColumnMeta('Reinvestment End Date'),
])

In [ ]:
summary = data.groupby('Original Rating', as_index=False)[['feature_A', 'feature_B']].mean()
summary_schema = schema.filter(lambda col: col.name in summary.columns)

(
    ExcelExporter('fake_bond_trades.xlsx')
    .add_sheet('Bond Trades', data, schema=schema, wrap_header=True)
    # Several tables on one sheet: without start_row, each goes below the previous one, gap_rows blank rows apart
    .add_sheet('Summary', summary, schema=summary_schema, wrap_header=True)
    .add_sheet('Summary', data.head(10), schema=schema, wrap_header=True, gap_rows=2)
    # An explicit start_row / start_col places a table anywhere
    .add_sheet('Bond Trades Offset', data, schema=schema, include_header=False, start_col=2, start_row=20)
    .save()
)